# **Modelo clasificatorio de calidad de champiñones**

In [107]:
# Instalación de librerías básicas
!pip install torch torchvision pandas numpy matplotlib scikit-learn seaborn

# **CARGAR DATOS Y ESTRATIFICACIÓN**

In [108]:
import pandas as pd

def load_data_from_github():
    try:
        #URLs Raw de archivos de github
        dataset_url = "https://raw.githubusercontent.com/VivianaPM/DL-VisNIR-QualityClassifier/refs/heads/feature/model-mushroom/Mushroom/Data/Dataset_Mus.csv"
        rangos_url = "https://raw.githubusercontent.com/VivianaPM/DL-VisNIR-QualityClassifier/refs/heads/feature/model-mushroom/Mushroom/Data/MushQualTags.csv"

        print("Cargando datos...")
        
        dataset = pd.read_csv(dataset_url, sep=',')
        rangos = pd.read_csv(rangos_url, sep=';')
        
        print(f"Dataset cargado: {dataset.shape}")
        print(f"Rangos cargados: {rangos.shape}")
        
        return dataset, rangos
        
    except Exception as e:
        print(f"Error: {e}")
        return None, None

## CARGAR DATOS DESDE GITHUB

In [109]:
df_Mush, rank_Mush = load_data_from_github()

if 'Dry matter' not in df_Mush.columns:
    print("Busca la columna equivalente a 'Dry matter' en:")
    print(df_Mush.columns.to_list())

Cargando datos...
Dataset cargado: (250, 206)
Rangos cargados: (6, 3)


In [110]:
display(rank_Mush)

,Category,Min_DryM,Max_DryM
0,Excelente,0.335,0.970
1,Muy buena,0.180,0.335
2,Buena,0.125,0.180
3,Razonable,0.089,0.125
4,Mala calidad,0.080,0.089
5,Muy mala,0.000,0.080


## ASIGNAR CATEGORÍA DEL MUSHQUALTAGS.CSV

In [111]:
def define_categories_mush(dry_matter, rank_Mush):
    for _, fila in rank_Mush.iterrows():
        if fila['Min_DryM'] <= dry_matter <= fila['Max_DryM']:
            return fila['Category']
    return 'Desconocida'

df_Mush['Category'] = df_Mush['Dry matter'].apply(
    lambda x: define_categories_mush(x, rank_Mush)
)

## DISTRIBUCIÓN DE CATEGORIAS

In [112]:
print(df_Mush['Category'].value_counts())

Category
Excelente       63
Razonable       62
Muy mala        51
Muy buena       36
Buena           26
Mala calidad    12
Name: count, dtype: int64


## CODIFICACIÓN TEMPORAL PARA ESTRATIFICAR

In [113]:
category_map_mush = {
    'Excelente': 0,
    'Razonable': 1,
    'Muy mala': 2,
    'Muy buena': 3,
    'Buena': 4,
    'Mala calidad': 5

}

df_Mush['strat_label'] = df_Mush['Category'].map(category_map_mush)

## Definición de X y Y globales

In [114]:
X = df_Mush.drop(columns=['Category', 'strat_label'])
y = df_Mush['strat_label']

## CHEKPOINT: REVISAR CODIFICACIÓN TEMPORAL

In [115]:
print(df_Mush[['Category', 'strat_label']].head())

       Category  strat_label
0  Mala calidad            5
1      Muy mala            2
2      Muy mala            2
3      Muy mala            2
4      Muy mala            2


# **DEFINICIÓN DE LA ARQUITECTURA DEL MLP**

In [116]:
import torch.nn as nn

class MLP_Mush(nn.Module):
    def __init__(self, input_size, hidden_layers, outputs_size, activation, dropout_rate=0.2):
        super().__init__()

        layers = []
        prev_size = input_size

        # Función de activación
        act_map = {
            'relu': nn.ReLU(),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid()
        }
        
        act = act_map[activation]

        # Hidden layers
        for h in hidden_layers:
            layers.append(nn.Linear(prev_size, h))
            layers.append(act)
            layers.append(nn.Dropout(dropout_rate))
            prev_size = h
        
        layers.append(nn.Linear(prev_size, outputs_size))

        self.network = nn.Sequential(*layers)

        def foward(self, x):
            return self.network(x)
        

# **STRATIFIED K-FOLD**

In [117]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## CHECKPOINT: VALIDACIÓN DE FOLDS

In [118]:
for i, (train_idx, val_idx) in enumerate(
        skf.split(df_Mush, df_Mush['strat_label'])):

    print(f"\nFOLD {i+1}")
    print("TRAIN", df_Mush.iloc[train_idx]['Category'].value_counts())
    print("VAL", df_Mush.iloc[val_idx]['Category'].value_counts())



FOLD 1
TRAIN Category
Excelente       51
Razonable       50
Muy mala        41
Muy buena       29
Buena           20
Mala calidad     9
Name: count, dtype: int64
VAL Category
Excelente       12
Razonable       12
Muy mala        10
Muy buena        7
Buena            6
Mala calidad     3
Name: count, dtype: int64

FOLD 2
TRAIN Category
Excelente       51
Razonable       50
Muy mala        41
Muy buena       28
Buena           21
Mala calidad     9
Name: count, dtype: int64
VAL Category
Razonable       12
Excelente       12
Muy mala        10
Muy buena        8
Buena            5
Mala calidad     3
Name: count, dtype: int64

FOLD 3
TRAIN Category
Excelente       50
Razonable       50
Muy mala        40
Muy buena       29
Buena           21
Mala calidad    10
Name: count, dtype: int64
VAL Category
Excelente       13
Razonable       12
Muy mala        11
Muy buena        7
Buena            5
Mala calidad     2
Name: count, dtype: int64

FOLD 4
TRAIN Category
Excelente       50
Razonable 

# **ENTRENAMIENTO Y OPTIMIZACIÓN DEL MODELO**

## Diccionario de Optimizadores

In [119]:
import torch.optim as optim 

OPTIMIZERS = {
    # ADAM
    'adam': lambda params, lr: optim.Adam(params, lr=lr),
    # SGD
    'sgd': lambda params, lr: optim.SGD(params, lr=lr, momentum=0.9),
    # RMSprop
    'rmsprop': lambda params, lr: optim.RMSprop(params, lr=lr),
    # AdamW
    'adamW': lambda params, lr: optim.AdamW(params, lr=lr) 
}

## Dicionario de funciones de perdida

In [120]:
CRITERIONS = {
    # Entropia cruzada
    'cross_entropy': nn.CrossEntropyLoss, #Recomendada segun lo que he probado

    'weighted_ce': lambda w=None: nn.CrossEntropyLoss(weight=w),
    '''
    La pérdida de probabilidad logarítmica negativa. Es útil entrenar una clasificación problema con C clases. 
    (NLLLoss - Pytorch2.9 documentation, 2023)
    '''
    'nll': nn.NLLLoss,  
}

## FUNCIÓN DE ENTRENAMIENTO DEL MLP

In [121]:
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

class Trainer:
    def __init__(self, model, device=None):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)

        self.history = {
            'train_loss': [],
            'train_accuracy': [],
            'val_loss': [],
            'val_accuracy': [],
            'val_precision': [],
            'val_recall': [],
            'val_f1': []            
        }

    def setup(self, 
                optimizer_name = 'adam', 
                criterion_name='cross_entropy', 
                lr=1e-4, 
                class_weight=None):
        # Optimizer
        if optimizer_name not in OPTIMIZERS:
            raise ValueError(f"Optimizador no soportado: {optimizer_name}")
        
        self.optimizer = OPTIMIZERS[optimizer_name](self.model.parameters(), lr)
        # Criterion
        if criterion_name == "weighted_ce":
            self.criterion = CRITERIONS["weighted_ce"](class_weight)
        else:
            self.criterion = CRITERIONS[criterion_name]()

    #funcion de entrenamiento por epoca - de aqui saco el loss y acc por epoca 
    def train_epochs(self, loader):
        self.model.train()

        total_loss = 0
        correct = 0
        total = 0

        for x, y in loader:
            x = x.to(self.device)
            y = y.to(self.device)

            self.optimizer.zero_grad()

            logits = self.model(x)
            loss = self.criterion(logits, y)

            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

            preds = logits.argmax(dim=1)

            correct += (preds == y).sum().item()
            total   += y.size(0)

        avg_loss = total_loss / len(loader)
        accuracy = correct / total

        return avg_loss, accuracy 

    #función de validación 
    def validate(self, loader):
        self.model.eval()

        total_loss = 0
        all_y = []
        all_pred = []

        with torch.no_grad():
            for x, y in loader():
                x, y = x.to(self.device), y.to(self.device)

                logits = self.model(x)
                loss = self.criterion(logits, y)

                preds = logits.argamax(dim=1)

                total_loss += loss.item()

                all_y.extend(y.cpu().numpy())
                all_pred.extend(preds.cpu().numpy())
        
        acc = accuracy_score(all_y, all_pred)
        prec = precision_score(all_y, all_pred, average='macro', zero_division=0)
        rec = recall_score(all_y, all_pred, average='macro', zero_division=0)
        f1 = f1_score(all_y, all_pred, average='macro', zero_division=0)

        return total_loss / len(loader), acc, prec, rec, f1

    # Función de entrenamiento
    def train(self, train_loader, val_loader, epochs=50):

            for epoch in range(epochs):

                train_loss, train_acc = self.train_epoch(train_loader)

                val_loss, acc, prec, rec, f1 = self.validate(val_loader)

                # Aqui guardo el historial de cada variable
                self.history['train_loss'].append(train_loss)
                self.history['train_accuracy'].append(train_acc)
                self.history['val_loss'].append(val_loss)
                self.history['val_accuracy'].append(acc)
                self.history['val_precision'].append(prec)
                self.history['val_recall'].append(rec)
                self.history['val_f1'].append(f1)

                if epoch % 10 == 0:
                    print(
                        f"Epoch {epoch:.3d} |"
                        f"Train: {train_loss:.3d} |"
                        f"Val: {val_loss:.3d} |"
                        f"Acc: {acc:.3d} |"
                        f"F1: {f1:.3d} |"
                    )
            
            return self.history

## LISTA PARA METRICAS POR FOLD

## DICCIONARIO DE OVERSAMPLERS

In [122]:
from imblearn.over_sampling import RandomOverSampler, SMOTE #instalar

OVERSAMPLERS = {
    "none": None, #Sin oversampling
    "ros": RandomOverSampler,
    'smote': SMOTE

}

In [123]:
def apply_oversampling(X, y, method="none", random_state=42):

    if method == "none":
        return X, y
    
    if method not in OVERSAMPLERS:
        raise ValueError(f"Oversampler no soportado {method}")
    
    sampler = OVERSAMPLERS[method](random_state=random_state)

    X_res, Y_res = sampler.fit_resample(X, y)

    return X_res, Y_res

In [124]:
MODEL_CONFIG = {
    "Modelo_Propuesta_1": {
        "hidden_layers": [32],
        "activation": 'relu',
        "dropout": 0.35,
        "epochs": 150,
        "oversamplers": 'ros',
        # función de optimización
        "optimizer":'adam', 
        "criterion": 'cross_entropy', 
        "lr": 1e-4, 
        "class_weight": None
    },

    # "Modelo_Base": {
    #     "hidden_layers": [128, 64],
    #     "dropout": 0.35
    # }  
}

In [125]:
from  torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
# Revisar lo que tengo en x y y
# print("X: ", X.shape)
# print("Y: ", y.shape)

for model_name, config in MODEL_CONFIG.items():

    print(f"EJECUTANDO {model_name}")
    # Lista de metricas para folds
    metrics_all_folds_mush = []
    train_losses_all = []
    val_losses_all = []

# loop del fold
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

        print(f"{'='*7} FOLD {fold} {'='*7}")

        #Paso #1: División del df_Mush
        X_train = X.iloc[train_idx]
        X_val   = X.iloc[val_idx]

        Y_train = y.iloc[train_idx]
        Y_val   = y.iloc[val_idx]  

        #Paso #2: Limpieza de las bandas
        spectral_bands = [c for c in X_train.columns if c not in ['Mush No']] 

        X_train_clean = X_train[spectral_bands]
        X_val_clean = X_val[spectral_bands]

        #Paso 4: Definición de los inputs de la red (Bandas espectrales)
        num_feactures = X_train_clean.shape[1]

        #Paso 5: Normalización - Buscando mejorar el rendimiento del modelo
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train_clean) 
        X_val_scaled = scaler.transform(X_val_clean)

        # Paso 6: OVERSAMPLING
        X_train_ros, Y_train_ros = apply_oversampling(
            X_train_scaled,
            Y_train,
            method=config['oversamplers']
        )  

        # Paso 7: Tensores
        X_train_tensor = torch.FloatTensor(X_train_ros)
        X_val_tensor = torch.FloatTensor(X_val_scaled)

        Y_train_tensor = torch.LongTensor(Y_train_ros.values)
        Y_val_tensor = torch.LongTensor(Y_val.values)

        # Paso 8: Creación de dataset y loader
        train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
        
        # Paso 9: Modelo + entrenamiento 
        model = MLP_Mush(
            input_size=num_feactures,
            hidden_layers=config['hidden_layers'],
            outputs_size=3,
            activation=config['activation'],       
            dropout_rate=config['dropout']    
        )

        trainer = Trainer(model)
        trainer.setup(config['optimizer'], config['criterion'], config['lr'], config['class_weight'])

        history = trainer.train(
            train_loader=train_loader,
            val_loader=val_loader,
            epochs = config['epochs']
        )

        # Paso 10: Guardar metricas
        for epoch in range(len(history['train_loss'])):

            metrics_all_folds_mush.append({
                "model": model_name,
                "fold": fold + 1,
                "epoch": epoch,

                "train_loss": history["train_loss"][epoch],
                "train_accuracy": history["train_accuracy"][epoch],

                "val_loss": history["val_loss"][epoch],
                "val_accuracy": history["val_accuracy"][epoch],
                "val_precision": history["val_precision"][epoch],
                "val_recall": history["val_recall"][epoch],
                "val_f1": history["val_f1"][epoch]               
            })

    

EJECUTANDO Modelo_Propuesta_1
======= FOLD 1 =======


AttributeError: 'Trainer' object has no attribute 'train_epoch'